# Systèmes de recommandation — Partie 2 : la librairie Surprise

**M2 — TP MovieLens (suite)**

On reprend le dataset `ml-latest-small` et on utilise cette fois la librairie [Surprise](https://surprise.readthedocs.io/) pour construire nos premiers systèmes de recommandation (aléatoire et "baseline"), qui serviront de référence de comparaison pour la suite du cours (user-based, item-based, matrix factorization...).

## 0. Installation

Si ce n'est pas déjà fait, installe la librairie dans ton environnement virtuel :
```bash
pip install scikit-surprise
```
⚠️ Le nom du package pip est `scikit-surprise`, mais on l'importe en Python avec `import surprise`.

In [3]:
import pandas as pd
import numpy as np

from surprise import Dataset, Reader
from surprise import NormalPredictor, BaselineOnly
from surprise.model_selection import train_test_split, cross_validate, KFold, LeaveOneOut, GridSearchCV
from surprise import accuracy

DATA_DIR = "../data"   # adapte le chemin si besoin
ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv")
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## 1. Étude de la documentation Surprise

Avant de coder, voici un résumé des notions clés (doc officielle : https://surprise.readthedocs.io/en/stable/getting_started.html).

### 1.1 — Chargement des données

Surprise ne travaille pas directement avec un DataFrame pandas : il faut passer par deux objets :

- **`Reader`** : décrit le format des notes. Pour un DataFrame, le seul paramètre obligatoire est `rating_scale=(min, max)` (ici `(0.5, 5)` pour MovieLens).
- **`Dataset.load_from_df(df, reader)`** : construit un objet `Dataset` à partir d'un DataFrame à **exactement 3 colonnes**, dans l'ordre : `user`, `item`, `rating`.

Il existe aussi `Dataset.load_builtin()` (datasets intégrés comme ml-100k) et `Dataset.load_from_file()` (fichier texte avec un `Reader` décrivant le format des lignes), mais dans notre cas on part de nos propres CSV déjà chargés avec pandas, donc `load_from_df` est le bon outil.

### 1.2 — Dataset, trainset et testset

C'est un point qui prête souvent à confusion dans Surprise :

- **`Dataset`** : l'ensemble brut des notes chargées, avec les identifiants "raw" (ceux de ton CSV).
- **`Trainset`** : structure interne optimisée pour l'entraînement (matrices creuses, ids internes "inner" différents des ids "raw"). On l'obtient via `train_test_split()`, un objet de validation croisée (`KFold`, `LeaveOneOut`...), ou directement tout le dataset avec `data.build_full_trainset()`.
- **`Testset`** : simple **liste de tuples** `(raw_user_id, raw_item_id, note_réelle)`. Ce n'est pas une structure optimisée comme le trainset — c'est juste ce sur quoi on appelle `algo.test(testset)` pour obtenir les prédictions.

Le split se fait comme en scikit-learn :
```python
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
algo.fit(trainset)
predictions = algo.test(testset)
```

### 1.3 — Validation croisée et Leave-One-Out

Le sous-module `surprise.model_selection` fournit les mêmes outils que scikit-learn :

- **`cross_validate(algo, data, measures=['RMSE','MAE'], cv=5)`** : validation croisée classique en K folds, retourne les scores par fold + moyenne + écart-type.
- **`KFold(n_splits=5)`** : itérateur bas niveau equivalent, utile si on veut boucler manuellement.
- **`GridSearchCV(algo_class, param_grid, measures=['rmse','mae'], cv=5)`** : recherche des meilleurs hyperparamètres par grille, comme `GridSearchCV` de scikit-learn.
- **`LeaveOneOut(n_splits=1, min_n_ratings=0)`** : ⚠️ **spécifique à Surprise**, différent du Leave-One-Out "générique". Ici on retire **une note par utilisateur** (pas une seule note au total) à chaque split — le jeu de test contient donc exactement une note par utilisateur. C'est pensé pour évaluer des recommandations top-N par utilisateur. Le paramètre `min_n_ratings` permet d'exclure les utilisateurs ayant trop peu de notes pour qu'un split ait du sens.
```python
loo = LeaveOneOut(n_splits=1, min_n_ratings=5, random_state=42)
for trainset, testset in loo.split(data):
    algo.fit(trainset)
    predictions = algo.test(testset)
```

## 2. Construction du dataset Surprise (matrice utilisateurs x films)

In [4]:
# Notes de 0.5 à 5.0 (par pas de 0.5) dans MovieLens
reader = Reader(rating_scale=(0.5, 5.0))

# Colonnes dans l'ordre attendu : user, item, rating
data = Dataset.load_from_df(ratings[["userId", "movieId", "rating"]], reader)

print(type(data))
print("Nombre de notes chargées :", len(data.raw_ratings))

<class 'surprise.dataset.DatasetAutoFolds'>
Nombre de notes chargées : 100836


In [5]:
# Un unique split train/test qu'on réutilisera pour comparer nos deux modèles équitablement
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print(f"Trainset : {trainset.n_ratings} notes, {trainset.n_users} utilisateurs, {trainset.n_items} films")
print(f"Testset  : {len(testset)} notes")

Trainset : 80668 notes, 610 utilisateurs, 8928 films
Testset  : 20168 notes


## 3. Système de recommandation aléatoire

Dans Surprise, le prédicteur aléatoire s'appelle **`NormalPredictor`** : il prédit une note tirée aléatoirement selon une loi normale dont la moyenne et l'écart-type sont estimés à partir de la distribution des notes du trainset. Il n'a **aucun hyperparamètre** à régler — on se contente donc d'un simple split train/test, sans validation croisée ni recherche d'hyperparamètres.

In [6]:
algo_random = NormalPredictor()
algo_random.fit(trainset)
predictions_random = algo_random.test(testset)

rmse_random = accuracy.rmse(predictions_random, verbose=True)
mae_random = accuracy.mae(predictions_random, verbose=True)

print(f"\nModèle ALÉATOIRE  ->  RMSE = {rmse_random:.4f}  |  MAE = {mae_random:.4f}")

RMSE: 1.4221
MAE:  1.1352

Modèle ALÉATOIRE  ->  RMSE = 1.4221  |  MAE = 1.1352


On s'attend à un RMSE/MAE assez élevé : ce modèle ne "comprend" rien, il tire juste une note au hasard autour de la distribution moyenne. C'est notre pire cas de référence — n'importe quel modèle un minimum sérieux devra faire mieux.

## 4. Système de recommandation "baseline"

Le modèle **`BaselineOnly`** prédit une note comme :

$$\hat{r}_{ui} = \mu + b_u + b_i$$

où $\mu$ est la moyenne globale des notes, $b_u$ le "biais" de l'utilisateur (note-t-il plutôt sévèrement ou généreusement ?) et $b_i$ le biais du film (est-il globalement mieux ou moins bien noté que la moyenne ?). C'est le modèle juste au-dessus du hasard : il ne fait aucune recommandation personnalisée fine, mais capture déjà les tendances générales.

Ses hyperparamètres dépendent de la méthode d'estimation choisie (`bsl_options`) :
- **méthode ALS** (par défaut) : `reg_i`, `reg_u`, `n_epochs`
- **méthode SGD** : `reg`, `learning_rate`, `n_epochs`

On cherche les meilleurs hyperparamètres par validation croisée avec `GridSearchCV`.

In [ ]:
param_grid = {
    "bsl_options": {
        "method": ["als", "sgd"],
        "n_epochs": [5, 10, 20],
        "reg_u": [5, 12, 20],      
        "reg_i": [5, 10, 20],      
    }
}

gs = GridSearchCV(BaselineOnly, param_grid, measures=["rmse", "mae"], cv=5, joblib_verbose=1)
gs.fit(data)

print("Meilleur RMSE (cross-validation) :", gs.best_score["rmse"])
print("Meilleurs paramètres (RMSE)      :", gs.best_params["rmse"])
print("\nMeilleur MAE (cross-validation)  :", gs.best_score["mae"])
print("Meilleurs paramètres (MAE)       :", gs.best_params["mae"])

Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimati

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    6.2s


Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimati

[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:   45.9s


Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimating biases using sgd...
Estimati

[Parallel(n_jobs=1)]: Done 270 out of 270 | elapsed:  1.1min finished


On récupère les meilleurs hyperparamètres trouvés par validation croisée, puis on entraîne le modèle final sur **le même split train/test** que le modèle aléatoire, pour pouvoir comparer les deux équitablement sur les mêmes données.

In [8]:
best_params = gs.best_params["rmse"]
algo_baseline = BaselineOnly(bsl_options=best_params["bsl_options"])
algo_baseline.fit(trainset)
predictions_baseline = algo_baseline.test(testset)

rmse_baseline = accuracy.rmse(predictions_baseline, verbose=True)
mae_baseline = accuracy.mae(predictions_baseline, verbose=True)

print(f"\nModèle BASELINE  ->  RMSE = {rmse_baseline:.4f}  |  MAE = {mae_baseline:.4f}")

Estimating biases using als...
RMSE: 0.8727
MAE:  0.6714

Modèle BASELINE  ->  RMSE = 0.8727  |  MAE = 0.6714


## Bilan comparatif

On rassemble les deux résultats dans un petit tableau récapitulatif — c'est cette table qu'on complétera au fil du cours avec les modèles user-based, item-based et matrix factorization.

In [9]:
comparaison = pd.DataFrame({
    "Modèle": ["Aléatoire (NormalPredictor)", "Baseline (BaselineOnly)"],
    "RMSE": [rmse_random, rmse_baseline],
    "MAE": [mae_random, mae_baseline],
})
comparaison

,Modèle,RMSE,MAE
0,Aléatoire (NormalPredictor),1.422058,1.135164
1,Baseline (BaselineOnly),0.872659,0.671352


**Ce qu'on doit observer :** le modèle baseline doit avoir un RMSE et un MAE nettement plus bas que le modèle aléatoire — la logique "moyenne globale + biais utilisateur + biais film" capture déjà beaucoup plus d'information que le pur hasard, même sans aucune notion de similarité entre utilisateurs ou entre films. C'est ce point de référence qu'on cherchera à battre avec les approches user-based / item-based / matrix factorization dans les prochains TP.